# ML Exploration

Prototype and benchmark regression models before moving production code to `model/train.py`.

**Goal:** predict next-day stock return (continuous float) → convert to BUY / SELL / HOLD.

**Data source:** `data/processed/{ticker}.csv` (output of `etl/etl.py`)

**Models compared:**
- LinearRegression — interpretable baseline (requires StandardScaler for Market_Cap)
- RandomForestRegressor — captures non-linear interactions, scale-invariant
- GradientBoostingRegressor — gradient-boosted trees, typically best on tabular data

**Metrics:** MAE, RMSE, R²  (regression metrics — NOT accuracy/AUC)

**Signal conversion (post-prediction):**
| Predicted return | Signal |
|---|---|
| > 0 | BUY  (1) |
| < 0 | SELL (-1) |
| = 0 | HOLD (0) |

In [ ]:
import os
import glob
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
print("Imports OK")

## 1. Feature definition

All 16 features produced by `etl/etl.py`.
**These must match `FEATURE_COLS` in `app/pages/go_live.py` and `backtesting.py`.**

In [ ]:
FEATURE_COLS = [
    # Price-based (6)
    "MA5", "MA20", "Volume_Change", "Market_Cap", "RSI", "MACD",
    # Volatility-normalised (5)
    "Log_Return", "Volatility_20", "Return_norm", "Return_norm_Lag1", "Return_norm_Lag2",
    # Fundamental (5) — quarterly, point-in-time merged
    "Gross_Margin", "Operating_Margin", "Net_Margin",
    "Debt_to_Equity", "Operating_CF_Ratio",
]
TARGET_COL = "Target"

print(f"Feature count : {len(FEATURE_COLS)}")
print(f"Features      : {FEATURE_COLS}")


## 2. Load processed data

Tickers are discovered dynamically from `data/processed/` — no hardcoded list.
Any ticker that has a processed CSV will be included automatically.

In [ ]:
PROCESSED_DIR = "../data/processed"

# Discover all available processed CSVs
csv_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*.csv")))
TICKERS = [os.path.basename(f).replace(".csv", "") for f in csv_files]

print(f"Found {len(TICKERS)} ticker(s): {TICKERS}")

# Load all into a dict — key = ticker symbol
dataframes = {}
for ticker in TICKERS:
    df = pd.read_csv(
        os.path.join(PROCESSED_DIR, f"{ticker}.csv"),
        parse_dates=["Date"],
    ).sort_values("Date").reset_index(drop=True)
    dataframes[ticker] = df
    n_fund_nan = df["Gross_Margin"].isna().sum()
    print(f"  {ticker}: {len(df)} rows | fundamental NaN: {n_fund_nan}")

## 3. Train / test split

80 / 20 **temporal** split — ordered by time, **never shuffled**.
Shuffling would leak future prices into training and produce falsely optimistic metrics.
The last 20 % of rows (most recent dates) are held out as the unseen test set.

In [ ]:
def time_split(df, feature_cols, target_col, train_ratio=0.80):
    """Return X_train, X_test, y_train, y_test using a temporal 80/20 split."""
    split_idx = int(len(df) * train_ratio)
    train = df.iloc[:split_idx].dropna(subset=feature_cols + [target_col])
    test  = df.iloc[split_idx:].dropna(subset=feature_cols + [target_col])

    return (
        train[feature_cols], test[feature_cols],
        train[target_col],   test[target_col],
    )

# Quick sanity check on the first ticker
t0 = TICKERS[0]
X_tr, X_te, y_tr, y_te = time_split(dataframes[t0], FEATURE_COLS, TARGET_COL)
df0 = dataframes[t0]
split_idx = int(len(df0) * 0.8)
print(f"[{t0}] Train: {len(X_tr)} rows  up to {df0.iloc[split_idx - 1]['Date'].date()}")
print(f"[{t0}] Test : {len(X_te)} rows  from {df0.iloc[split_idx]['Date'].date()}")

## 4. Evaluation helper

Three regression metrics:
- **MAE** (Mean Absolute Error) — average magnitude of error; same unit as the return
- **RMSE** (Root Mean Squared Error) — penalises large errors more than MAE
- **R²** — fraction of variance explained (1.0 = perfect, 0.0 = just predicting the mean)

In [ ]:
def evaluate(name, y_true, y_pred):
    """Compute and print MAE, RMSE, R2. Returns a result dict."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2   = r2_score(y_true, y_pred)
    print(f"  {name:<30s}  MAE={mae:.5f}  RMSE={rmse:.5f}  R2={r2:+.4f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

## 5. Model training & evaluation

Train all three models on every ticker and collect metrics.

**Why StandardScaler for LinearRegression?**
`Market_Cap` is ~10¹² while `Return` is ~0.01 — a difference of 14 orders of magnitude.
Without scaling, the linear solver is dominated by Market_Cap and the coefficients for
smaller features are effectively zero. Tree-based models split on thresholds and are
unaffected by scale, so no scaling is needed for RF or GBR.

In [ ]:
def build_pipelines():
    """Return a fresh dict of named sklearn Pipelines."""
    return {
        "LinearRegression": Pipeline([
            ("scaler", StandardScaler()),   # mandatory — Market_Cap dominates without it
            ("reg",    LinearRegression()),
        ]),
        "RandomForest": Pipeline([
            # Scale-invariant; n_jobs=-1 uses all CPU cores
            ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ]),
        "GradientBoosting": Pipeline([
            # Best on tabular data; slower than RF
            ("reg", GradientBoostingRegressor(n_estimators=200, random_state=42)),
        ]),
    }

all_results = []

for ticker in TICKERS:
    df = dataframes[ticker]
    X_train, X_test, y_train, y_test = time_split(df, FEATURE_COLS, TARGET_COL)
    print(f"\n{'='*58}")
    print(f"  {ticker}  (train={len(X_train)}, test={len(X_test)})")
    print(f"{'='*58}")

    for name, pipeline in build_pipelines().items():
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)
        row = evaluate(name, y_test, preds)
        row["ticker"] = ticker
        all_results.append(row)

results_df = pd.DataFrame(all_results)[["ticker", "model", "MAE", "RMSE", "R2"]]
print("\n\nSummary across all tickers:")
print(results_df.to_string(index=False))

## 6. Signal conversion & direction accuracy

The model outputs a continuous return estimate.
We convert to a trading signal by checking the **sign**:

```
predicted return > 0  →  BUY  (model expects price to rise)
predicted return < 0  →  SELL (model expects price to fall)
predicted return = 0  →  HOLD
```

**Direction accuracy** = fraction of test rows where `sign(predicted) == sign(actual)`.
This is more meaningful for a trading system than R².

In [ ]:
def direction_accuracy(y_true, y_pred):
    """Fraction of predictions with the correct sign (direction)."""
    return (np.sign(y_pred) == np.sign(y_true)).mean()

print(f"{'Ticker':<8} {'Model':<22} {'Dir. Acc':>10}")
print("-" * 42)

for ticker in TICKERS:
    df = dataframes[ticker]
    X_train, X_test, y_train, y_test = time_split(df, FEATURE_COLS, TARGET_COL)
    for name, pipeline in build_pipelines().items():
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)
        acc = direction_accuracy(y_test, preds)
        print(f"{ticker:<8} {name:<22} {acc:>10.2%}")

## 7. Feature importance (Random Forest — first ticker)

Shows which of the 17 features the RF model relies on most.
Useful for understanding model behaviour and for future feature pruning.

In [ ]:
t0 = TICKERS[0]
df = dataframes[t0]
X_train, X_test, y_train, y_test = time_split(df, FEATURE_COLS, TARGET_COL)

rf_pipe = Pipeline([
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)

importances = rf_pipe.named_steps["reg"].feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
feat_imp.plot(kind="bar", ax=ax, color="#00c805")
ax.set_title(f"RF Feature Importance — {t0}", fontsize=13)
ax.set_ylabel("Importance")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

print("\nTop features:")
print(feat_imp.head(10).to_string())

## 8. Export best model

Re-train the best model on the **full dataset** (train + test combined) for each ticker,
then save to `model/trained/model_{ticker}.pkl`.

Using the full dataset for the final export is standard practice — the test split was
only needed for unbiased evaluation. The production model benefits from seeing all
available history.

In [ ]:
TRAINED_DIR = "../model/trained"
os.makedirs(TRAINED_DIR, exist_ok=True)

# Set to whichever model had the best R2 / direction accuracy in sections 5 and 6.
# GradientBoosting is usually strongest on tabular data; change if results differ.
BEST_MODEL_NAME = "GradientBoosting"

for ticker in TICKERS:
    df = dataframes[ticker].dropna(subset=FEATURE_COLS + [TARGET_COL])
    X_all = df[FEATURE_COLS]
    y_all = df[TARGET_COL]

    # Re-train on all data — no train/test split for final production model
    pipeline = build_pipelines()[BEST_MODEL_NAME]
    pipeline.fit(X_all, y_all)

    out_path = os.path.join(TRAINED_DIR, f"model_{ticker}.pkl")
    joblib.dump(pipeline, out_path)
    print(f"Saved  {ticker:6s}  →  {out_path}")

print(f"\nDone. {len(TICKERS)} model(s) exported using {BEST_MODEL_NAME}.")
print("Run  python model/train.py --all  to reproduce via CLI.")